Test influence of coarse and rerank model in model2 to the final result


In [1]:
!pip install -q datasets transformers huggingface_hub


In [2]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader

from datasets import load_dataset
from transformers import BertTokenizer, BertModel, ViTModel, ViTImageProcessor
from huggingface_hub import hf_hub_download

from tqdm.auto import tqdm


In [3]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(device)


cuda


In [4]:
dataset = load_dataset("Zoe3324/flickr30k-pairs")
test_data = dataset["test"]
print(f"test size: {len(test_data)}")


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md:   0%|          | 0.00/657 [00:00<?, ?B/s]

data/train-00000-of-00012.parquet:   0%|          | 0.00/97.0M [00:00<?, ?B/s]

data/train-00001-of-00012.parquet:   0%|          | 0.00/96.2M [00:00<?, ?B/s]

data/train-00002-of-00012.parquet:   0%|          | 0.00/101M [00:00<?, ?B/s]

...

data/test-00000-of-00001.parquet:   0%|          | 0.00/40.1M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/145000 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/5070 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/5000 [00:00<?, ? examples/s]

test size: 5000


In [5]:
tokenizer = BertTokenizer.from_pretrained("bert-base-uncased")
image_processor = ViTImageProcessor.from_pretrained("google/vit-base-patch16-224")


tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

preprocessor_config.json:   0%|          | 0.00/160 [00:00<?, ?B/s]

In [6]:
class CrossAttention(nn.Module):
    def __init__(self, dim):
        super().__init__()
        self.query = nn.Linear(dim, dim)
        self.key = nn.Linear(dim, dim)
        self.value = nn.Linear(dim, dim)
        self.scale = dim ** -0.5

    def forward(self, q, k, v):
        Q = self.query(q)
        K = self.key(k)
        V = self.value(v)
        attn = torch.matmul(Q, K.transpose(-2, -1)) * self.scale
        attn = torch.softmax(attn, dim=-1)
        out = torch.matmul(attn, V)
        return out


class MultiModalModel(nn.Module):
    def __init__(self, embed_dim=256):
        super().__init__()
        self.text_encoder = BertModel.from_pretrained("bert-base-uncased")
        self.image_encoder = ViTModel.from_pretrained("google/vit-base-patch16-224")

        dim = 768
        self.image_proj = nn.Linear(dim, embed_dim)
        self.text_proj = nn.Linear(dim, embed_dim)

        self.cross_attn = CrossAttention(dim)
        self.match_head = nn.Sequential(
            nn.Linear(dim, embed_dim),
            nn.ReLU(),
            nn.Linear(embed_dim, 1),
        )

        self.temperature = nn.Parameter(torch.tensor(0.07))

    def encode_image(self, pixel_values):
        image_out = self.image_encoder(pixel_values=pixel_values).last_hidden_state
        image_emb = F.normalize(self.image_proj(image_out[:, 0]), dim=-1)
        return image_emb, image_out

    def encode_text(self, input_ids, attention_mask):
        text_out = self.text_encoder(input_ids=input_ids, attention_mask=attention_mask).last_hidden_state
        text_emb = F.normalize(self.text_proj(text_out[:, 0]), dim=-1)
        return text_emb, text_out

    def compute_itm(self, text_tokens, image_tokens):
        fused = self.cross_attn(text_tokens, image_tokens, image_tokens)
        fused = fused.mean(dim=1)
        logits = self.match_head(fused).squeeze(-1)
        return logits

    def forward(self, input_ids, attention_mask, pixel_values):
        image_emb, image_tokens = self.encode_image(pixel_values)
        text_emb, text_tokens = self.encode_text(input_ids, attention_mask)
        itm_logits = self.compute_itm(text_tokens, image_tokens)
        return image_emb, text_emb, itm_logits


In [7]:
def eval_collate_grouped(batch):
    texts = [x["caption"] for x in batch]
    images = [x["image"].convert("RGB") for x in batch]
    img_ids = [x["img_id"] for x in batch]
    return texts, images, img_ids


In [8]:
def evaluate_image_to_text_grouped_fused(model, data, num_samples=200, top_k=20, alpha=1.0, beta=0.5):
    subset = data.select(range(min(num_samples, len(data))))
    loader = DataLoader(subset, batch_size=32, shuffle=False, collate_fn=eval_collate_grouped)

    all_texts = []
    all_images = []
    all_img_ids = []
    for texts, images, img_ids in loader:
        all_texts.extend(texts)
        all_images.extend(images)
        all_img_ids.extend(img_ids)

    group_to_indices = {}
    for idx, gid in enumerate(all_img_ids):
        if gid not in group_to_indices:
            group_to_indices[gid] = []
        group_to_indices[gid].append(idx)

    N = len(all_texts)
    all_image_embs = []
    all_text_embs = []
    model.eval()
    bs = 64

    with torch.no_grad():
        for start in range(0, N, bs):
            end = min(start + bs, N)
            image_enc = image_processor(images=all_images[start:end], return_tensors="pt")
            img_emb, _ = model.encode_image(image_enc["pixel_values"].to(device))
            all_image_embs.append(img_emb.cpu())

        for start in range(0, N, bs):
            end = min(start + bs, N)
            text_enc = tokenizer(all_texts[start:end], padding="max_length", truncation=True, max_length=32, return_tensors="pt")
            txt_emb, _ = model.encode_text(text_enc["input_ids"].to(device), text_enc["attention_mask"].to(device))
            all_text_embs.append(txt_emb.cpu())

    all_image_embs = torch.cat(all_image_embs)
    all_text_embs = torch.cat(all_text_embs)
    coarse_sim = all_image_embs @ all_text_embs.T

    rerank_matrix = torch.full((N, N), -1e9)
    fused_matrix = torch.full((N, N), -1e9)

    with torch.no_grad():
        for i in tqdm(range(N), desc="image-to-text fused"):
            candidates = coarse_sim[i].topk(top_k).indices.tolist()
            candidate_texts = [all_texts[j] for j in candidates]
            candidate_images = [all_images[i]] * len(candidates)

            text_enc = tokenizer(candidate_texts, padding="max_length", truncation=True, max_length=32, return_tensors="pt")
            image_enc = image_processor(images=candidate_images, return_tensors="pt")
            _, _, logits = model(
                text_enc["input_ids"].to(device),
                text_enc["attention_mask"].to(device),
                image_enc["pixel_values"].to(device),
            )

            rerank_scores = torch.sigmoid(logits).cpu()
            for local_idx, j in enumerate(candidates):
                rerank_matrix[i, j] = rerank_scores[local_idx]
                fused_matrix[i, j] = alpha * coarse_sim[i, j] + beta * rerank_scores[local_idx]

    def recall_at_k(score_mat, k):
        correct = 0
        for i, gid in enumerate(all_img_ids):
            correct_set = set(group_to_indices[gid])
            topk = set(score_mat[i].topk(k).indices.tolist())
            if topk & correct_set:
                correct += 1
        return correct / len(all_img_ids)

    return {
        "image_to_text_coarse_r1": recall_at_k(coarse_sim, 1),
        "image_to_text_coarse_r5": recall_at_k(coarse_sim, 5),
        "image_to_text_coarse_r10": recall_at_k(coarse_sim, 10),
        "image_to_text_rerank_r1": recall_at_k(rerank_matrix, 1),
        "image_to_text_rerank_r5": recall_at_k(rerank_matrix, 5),
        "image_to_text_rerank_r10": recall_at_k(rerank_matrix, 10),
        "image_to_text_fused_r1": recall_at_k(fused_matrix, 1),
        "image_to_text_fused_r5": recall_at_k(fused_matrix, 5),
        "image_to_text_fused_r10": recall_at_k(fused_matrix, 10),
    }


In [9]:
def evaluate_text_to_image_grouped_fused(model, data, num_samples=200, top_k=20, alpha=1.0, beta=0.5):
    subset = data.select(range(min(num_samples, len(data))))
    loader = DataLoader(subset, batch_size=32, shuffle=False, collate_fn=eval_collate_grouped)

    all_texts = []
    all_images = []
    all_img_ids = []
    for texts, images, img_ids in loader:
        all_texts.extend(texts)
        all_images.extend(images)
        all_img_ids.extend(img_ids)

    group_to_indices = {}
    for idx, gid in enumerate(all_img_ids):
        if gid not in group_to_indices:
            group_to_indices[gid] = []
        group_to_indices[gid].append(idx)

    N = len(all_texts)
    all_image_embs = []
    all_text_embs = []
    model.eval()
    bs = 64

    with torch.no_grad():
        for start in range(0, N, bs):
            end = min(start + bs, N)
            image_enc = image_processor(images=all_images[start:end], return_tensors="pt")
            img_emb, _ = model.encode_image(image_enc["pixel_values"].to(device))
            all_image_embs.append(img_emb.cpu())

        for start in range(0, N, bs):
            end = min(start + bs, N)
            text_enc = tokenizer(all_texts[start:end], padding="max_length", truncation=True, max_length=32, return_tensors="pt")
            txt_emb, _ = model.encode_text(text_enc["input_ids"].to(device), text_enc["attention_mask"].to(device))
            all_text_embs.append(txt_emb.cpu())

    all_image_embs = torch.cat(all_image_embs)
    all_text_embs = torch.cat(all_text_embs)
    coarse_sim = all_text_embs @ all_image_embs.T

    rerank_matrix = torch.full((N, N), -1e9)
    fused_matrix = torch.full((N, N), -1e9)

    with torch.no_grad():
        for i in tqdm(range(N), desc="text-to-image fused"):
            candidates = coarse_sim[i].topk(top_k).indices.tolist()
            candidate_images = [all_images[j] for j in candidates]
            candidate_texts = [all_texts[i]] * len(candidates)

            text_enc = tokenizer(candidate_texts, padding="max_length", truncation=True, max_length=32, return_tensors="pt")
            image_enc = image_processor(images=candidate_images, return_tensors="pt")
            _, _, logits = model(
                text_enc["input_ids"].to(device),
                text_enc["attention_mask"].to(device),
                image_enc["pixel_values"].to(device),
            )

            rerank_scores = torch.sigmoid(logits).cpu()
            for local_idx, j in enumerate(candidates):
                rerank_matrix[i, j] = rerank_scores[local_idx]
                fused_matrix[i, j] = alpha * coarse_sim[i, j] + beta * rerank_scores[local_idx]

    def recall_at_k(score_mat, k):
        correct = 0
        for i, gid in enumerate(all_img_ids):
            correct_set = set(group_to_indices[gid])
            topk = set(score_mat[i].topk(k).indices.tolist())
            if topk & correct_set:
                correct += 1
        return correct / len(all_img_ids)

    return {
        "text_to_image_coarse_r1": recall_at_k(coarse_sim, 1),
        "text_to_image_coarse_r5": recall_at_k(coarse_sim, 5),
        "text_to_image_coarse_r10": recall_at_k(coarse_sim, 10),
        "text_to_image_rerank_r1": recall_at_k(rerank_matrix, 1),
        "text_to_image_rerank_r5": recall_at_k(rerank_matrix, 5),
        "text_to_image_rerank_r10": recall_at_k(rerank_matrix, 10),
        "text_to_image_fused_r1": recall_at_k(fused_matrix, 1),
        "text_to_image_fused_r5": recall_at_k(fused_matrix, 5),
        "text_to_image_fused_r10": recall_at_k(fused_matrix, 10),
    }


In [10]:
repo_id = "qasxxsaq/Image-Text-Matching-Model-Advanced"
revision = "Similarity+Rerank"
filename = "advanced_model_weights_similarity+rerank.pt"

FUSION_ALPHA = 1.0
FUSION_BETA = 0.5
NUM_SAMPLES = 200
TOP_K = 20

ckpt_path = hf_hub_download(
    repo_id=repo_id,
    filename=filename,
    repo_type="model",
    revision=revision,
)

checkpoint = torch.load(ckpt_path, map_location=device, weights_only=False)

model = MultiModalModel().to(device)
model.load_state_dict(checkpoint)
model.eval()

print(f"Loaded checkpoint from {repo_id}/{filename} @ {revision}")
print(f"Fusion weights: alpha={FUSION_ALPHA}, beta={FUSION_BETA}")


advanced_model_weights_similarity+rerank(…):   0%|          | 0.00/793M [00:00<?, ?B/s]

config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/440M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: bert-base-uncased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/346M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/198 [00:00<?, ?it/s]

ViTModel LOAD REPORT from: google/vit-base-patch16-224
Key                 | Status     | 
--------------------+------------+-
classifier.weight   | UNEXPECTED | 
classifier.bias     | UNEXPECTED | 
pooler.dense.bias   | MISSING    | 
pooler.dense.weight | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Loaded checkpoint from qasxxsaq/Image-Text-Matching-Model-Advanced/advanced_model_weights_similarity+rerank.pt @ Similarity+Rerank
Fusion weights: alpha=1.0, beta=0.5


In [11]:
i2t_results = evaluate_image_to_text_grouped_fused(
    model,
    test_data,
    num_samples=NUM_SAMPLES,
    top_k=TOP_K,
    alpha=FUSION_ALPHA,
    beta=FUSION_BETA,
)

t2i_results = evaluate_text_to_image_grouped_fused(
    model,
    test_data,
    num_samples=NUM_SAMPLES,
    top_k=TOP_K,
    alpha=FUSION_ALPHA,
    beta=FUSION_BETA,
)

results = {**i2t_results, **t2i_results}
results["coarse_mean_recall"] = float((
    results["image_to_text_coarse_r1"] + results["image_to_text_coarse_r5"] + results["image_to_text_coarse_r10"] +
    results["text_to_image_coarse_r1"] + results["text_to_image_coarse_r5"] + results["text_to_image_coarse_r10"]
) / 6)
results["rerank_mean_recall"] = float((
    results["image_to_text_rerank_r1"] + results["image_to_text_rerank_r5"] + results["image_to_text_rerank_r10"] +
    results["text_to_image_rerank_r1"] + results["text_to_image_rerank_r5"] + results["text_to_image_rerank_r10"]
) / 6)
results["fused_mean_recall"] = float((
    results["image_to_text_fused_r1"] + results["image_to_text_fused_r5"] + results["image_to_text_fused_r10"] +
    results["text_to_image_fused_r1"] + results["text_to_image_fused_r5"] + results["text_to_image_fused_r10"]
) / 6)
results


image-to-text fused:   0%|          | 0/200 [00:00<?, ?it/s]

text-to-image fused:   0%|          | 0/200 [00:00<?, ?it/s]

{'image_to_text_coarse_r1': 0.925,
 'image_to_text_coarse_r5': 1.0,
 'image_to_text_coarse_r10': 1.0,
 'image_to_text_rerank_r1': 0.875,
 'image_to_text_rerank_r5': 0.975,
 'image_to_text_rerank_r10': 0.975,
 'image_to_text_fused_r1': 0.925,
 'image_to_text_fused_r5': 1.0,
 'image_to_text_fused_r10': 1.0,
 'text_to_image_coarse_r1': 0.865,
 'text_to_image_coarse_r5': 0.865,
 'text_to_image_coarse_r10': 0.935,
 'text_to_image_rerank_r1': 0.805,
 'text_to_image_rerank_r5': 0.805,
 'text_to_image_rerank_r10': 0.94,
 'text_to_image_fused_r1': 0.87,
 'text_to_image_fused_r5': 0.87,
 'text_to_image_fused_r10': 0.92,
 'coarse_mean_recall': 0.9316666666666666,
 'rerank_mean_recall': 0.8958333333333334,
 'fused_mean_recall': 0.9308333333333333}

In [13]:
# Optional: try a few fusion weights without changing anything else
settings = [
    (1.0, 0.0),
    (1.0, 0.2),
    (1.0, 0.5),
    (1.0, 1.0),
]

all_results = {}
for alpha, beta in settings:
    print(f"running alpha={alpha}, beta={beta}")
    i2t_results = evaluate_image_to_text_grouped_fused(model, test_data, num_samples=NUM_SAMPLES, top_k=TOP_K, alpha=alpha, beta=beta)
    t2i_results = evaluate_text_to_image_grouped_fused(model, test_data, num_samples=NUM_SAMPLES, top_k=TOP_K, alpha=alpha, beta=beta)
    merged = {**i2t_results, **t2i_results}
    merged["coarse_mean_recall"] = float((
        merged["image_to_text_coarse_r1"] + merged["image_to_text_coarse_r5"] + merged["image_to_text_coarse_r10"] +
        merged["text_to_image_coarse_r1"] + merged["text_to_image_coarse_r5"] + merged["text_to_image_coarse_r10"]
    ) / 6)
    merged["rerank_mean_recall"] = float((
        merged["image_to_text_rerank_r1"] + merged["image_to_text_rerank_r5"] + merged["image_to_text_rerank_r10"] +
        merged["text_to_image_rerank_r1"] + merged["text_to_image_rerank_r5"] + merged["text_to_image_rerank_r10"]
    ) / 6)
    merged["fused_mean_recall"] = float((
        merged["image_to_text_fused_r1"] + merged["image_to_text_fused_r5"] + merged["image_to_text_fused_r10"] +
        merged["text_to_image_fused_r1"] + merged["text_to_image_fused_r5"] + merged["text_to_image_fused_r10"]
    ) / 6)
    all_results[(alpha, beta)] = merged

all_results


running alpha=1.0, beta=0.0


image-to-text fused:   0%|          | 0/200 [00:00<?, ?it/s]

text-to-image fused:   0%|          | 0/200 [00:00<?, ?it/s]

running alpha=1.0, beta=0.2


image-to-text fused:   0%|          | 0/200 [00:00<?, ?it/s]

...

text-to-image fused:   0%|          | 0/200 [00:00<?, ?it/s]

running alpha=1.0, beta=1.0


image-to-text fused:   0%|          | 0/200 [00:00<?, ?it/s]

text-to-image fused:   0%|          | 0/200 [00:00<?, ?it/s]

{(1.0, 0.0): {'image_to_text_coarse_r1': 0.925,
  'image_to_text_coarse_r5': 1.0,
  'image_to_text_coarse_r10': 1.0,
  'image_to_text_rerank_r1': 0.875,
  'image_to_text_rerank_r5': 0.975,
  'image_to_text_rerank_r10': 0.975,
  'image_to_text_fused_r1': 0.925,
  'image_to_text_fused_r5': 1.0,
  'image_to_text_fused_r10': 1.0,
  'text_to_image_coarse_r1': 0.865,
  'text_to_image_coarse_r5': 0.865,
  'text_to_image_coarse_r10': 0.935,
  'text_to_image_rerank_r1': 0.805,
  'text_to_image_rerank_r5': 0.805,
  'text_to_image_rerank_r10': 0.94,
  'text_to_image_fused_r1': 0.865,
  'text_to_image_fused_r5': 0.865,
  'text_to_image_fused_r10': 0.935,
  'coarse_mean_recall': 0.9316666666666666,
  'rerank_mean_recall': 0.8958333333333334,
  'fused_mean_recall': 0.9316666666666666},
 (1.0, 0.2): {'image_to_text_coarse_r1': 0.925,
  'image_to_text_coarse_r5': 1.0,
  'image_to_text_coarse_r10': 1.0,
  'image_to_text_rerank_r1': 0.875,
  'image_to_text_rerank_r5': 0.975,
  'image_to_text_rerank_r10'